# Task 2 v4 - Task 1 Candidate Industries + Task 2 Cross-Encoder Reranker

Goal:

1. Use a Task 1 industry model to generate top parent-industry candidates.
2. Expand those parent industries into possible 10-digit subindustries using the taxonomy.
3. Use the trained Task 2 cross-encoder to rank those candidate subindustries.

Important design rule:

- Task 1 may use `LongProfile` to find likely parent industries.
- Task 2 ranking uses only `SegmentName`, `SegmentDescription`, and candidate taxonomy text.

In [1]:
from pathlib import Path
import gc
import html
import json
import math
import os
import random
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
from scipy.special import log_softmax, softmax
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)

## Config

In [4]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

BASE_DIR     = Path('/content/drive/MyDrive/CAPSTONE')
ART_DIR_V11  = BASE_DIR / 'flangbert_v10_artifacts'
T2_ASSETS    = BASE_DIR / 'task2_assets'

SEED = 42

TASK2_PATH        = T2_ASSETS / 'task2_subindustry_classification_final.csv'
TASK1_SOURCE_PATH = BASE_DIR / 'cleaned_v10' / 'task1_gecs_cleaned_v10.csv'  # update if different location
TAXONOMY_PATH     = T2_ASSETS / 'taxonomy_table.csv'
SPLIT_PATH        = T2_ASSETS / 'split_assignments.csv'

# v11 FLANG-BERT weights
TASK1_V11_STATE      = ART_DIR_V11 / 'best_model_state.pt'
TASK1_V11_LABEL_ENC  = ART_DIR_V11 / 'label_encoder.pkl'
TASK1_V11_SECTOR_ENC = ART_DIR_V11 / 'label_encoder_sector.pkl'
TASK1_V11_GROUP_ENC  = ART_DIR_V11 / 'label_encoder_group.pkl'
TASK1_V11_SCALER     = ART_DIR_V11 / 'numeric_scaler.pkl'

CROSS_ENCODER_DIR = T2_ASSETS / 'cross_encoder'
OUTPUT_DIR        = Path('/content/drive/MyDrive/CAPSTONE/task2_v11_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK2_CROSS_ENCODER_MODEL = "microsoft/deberta-v3-small"
TASK1_V11_MODEL_NAME      = "SALT-NLP/FLANG-BERT"
LOCAL_FILES_ONLY          = False

PARENT_TOPK              = 10
MIN_CANDIDATE_LEAFS      = 0
LOW_CONFIDENCE_THRESHOLD = 0.35
FALLBACK_PARENT_TOPK     = 15

EVAL_ROW_LIMIT  = None
MAX_LENGTH      = 256
EVAL_BATCH_SIZE = 16

ALPHA_GRID = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 7.5, 10.0, 15.0]

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS    = 1e-12

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

print("device:", DEVICE)
print()
print("=== File check ===")
for p in [TASK2_PATH, TASK1_SOURCE_PATH, TAXONOMY_PATH, SPLIT_PATH,
          TASK1_V11_STATE, TASK1_V11_LABEL_ENC, TASK1_V11_SECTOR_ENC,
          TASK1_V11_GROUP_ENC, TASK1_V11_SCALER,
          CROSS_ENCODER_DIR / 'best_state.pt']:
    print(" ", "OK     " if p.exists() else "MISSING", p)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda

=== File check ===
  OK      /content/drive/MyDrive/CAPSTONE/task2_assets/task2_subindustry_classification_final.csv
  OK      /content/drive/MyDrive/CAPSTONE/cleaned_v10/task1_gecs_cleaned_v10.csv
  OK      /content/drive/MyDrive/CAPSTONE/task2_assets/taxonomy_table.csv
  OK      /content/drive/MyDrive/CAPSTONE/task2_assets/split_assignments.csv
  OK      /content/drive/MyDrive/CAPSTONE/flangbert_v10_artifacts/best_model_state.pt
  OK      /content/drive/MyDrive/CAPSTONE/flangbert_v10_artifacts/label_encoder.pkl
  OK      /content/drive/MyDrive/CAPSTONE/flangbert_v10_artifacts/label_encoder_sector.pkl
  OK      /content/drive/MyDrive/CAPSTONE/flangbert_v10_artifacts/label_encoder_group.pkl
  OK      /content/drive/MyDrive/CAPSTONE/flangbert_v10_artifacts/numeric_scaler.pkl
  MISSING /content/drive/MyDrive/CAPSTONE/task2_assets/cross_encoder/bes

In [5]:
import shutil
from pathlib import Path

src = Path('/content/drive/MyDrive/CAPSTONE/task2_assets/cross_encoder/best_state')
dst = Path('/content/drive/MyDrive/CAPSTONE/task2_assets/cross_encoder/best_state.pt')

if src.exists() and not dst.exists():
    shutil.copy(src, dst)
    print("Renamed — best_state.pt now exists")
elif dst.exists():
    print("Already correct — best_state.pt exists")
else:
    print("ERROR: source file not found either")

Already correct — best_state.pt exists


## Text and Metric Helpers

In [6]:
SMART_TRANS = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-", "\u2212": "-", "\xa0": " ", "\ufeff": ""
})
WS_RE = re.compile(r"\s+")
URL_RE = re.compile(r"http\S+|www\.\S+")
NUMBER_RE = re.compile(r"\b\d[\d,\.]*\b")
NON_ALPHA_RE = re.compile(r"[^a-z\s]")


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value)).translate(SMART_TRANS)
    return WS_RE.sub(" ", text).strip()


def clean_code(value, width=None):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    digits = re.sub(r"\D+", "", text)
    if width and digits:
        return digits.zfill(width)
    return digits


def light_clean(value):
    text = normalize_text(value).lower()
    text = URL_RE.sub(" ", text)
    return WS_RE.sub(" ", text).strip()


DOMAIN_STOPWORDS = {
    "company", "companies", "firm", "firms", "corporation", "corp", "inc", "ltd", "plc", "sa",
    "segment", "segments", "business", "businesses", "operation", "operations", "operating",
    "service", "services", "product", "products", "provide", "provides", "providing", "provided",
    "offer", "offers", "offering", "offered", "include", "includes", "including", "included",
    "primarily", "geographically", "revenue", "revenues", "generate", "generates", "generated",
    "generating", "derive", "derives", "derived", "deriving", "also", "well", "various",
    "mainly", "majority", "largest", "main", "across", "engage", "engaged", "engages",
    "engaging", "consist", "consists", "consisting", "comprise", "comprises", "manage",
    "manages", "managed", "managing", "management", "produce", "produces", "produced",
    "producing", "production", "develop", "develops", "developed", "developing", "development",
    "manufacture", "manufactures", "manufactured", "manufacturing", "manufacturer", "distribute",
    "distributes", "distributed", "distributing", "distribution", "distributor", "market",
    "markets", "marketed", "marketing", "sell", "sells", "sold", "selling", "sale", "sales",
    "use", "uses", "used", "using", "addition", "additional", "additionally", "global",
    "globally", "worldwide", "international", "domestic", "year", "years", "annual", "quarterly",
    "report", "reports", "reportable", "reported", "reporting", "customer", "customers",
    "client", "clients", "region", "regions", "regional", "area", "areas", "country",
    "countries", "north", "south", "east", "west", "american", "america", "europe",
    "european", "asia", "asian", "africa", "african", "pacific", "china", "chinese",
    "japan", "japanese", "india", "indian", "united", "states", "world", "based",
    "primary", "key", "large", "small", "new", "high", "low", "approximately", "per",
    "cent", "percent", "million", "billion", "thousand",
}
BOILERPLATE_PHRASES = [
    "the company is", "the company has", "the company also", "geographically the company",
    "the company generates", "the company derives", "the company provides", "the company operates",
    "the company offers", "the company manufactures", "the company s main",
    "the company s reportable", "the company s operating segment", "the company s operating segments",
    "the firm is", "the firm has", "as well as", "rest of the world", "rest of world",
    "north america", "south america", "asia pacific", "middle east", "latin america",
]


def heavy_clean(value):
    text = light_clean(value)
    text = NUMBER_RE.sub(" ", text)
    text = NON_ALPHA_RE.sub(" ", text)
    for phrase in BOILERPLATE_PHRASES:
        text = text.replace(phrase, " ")
    tokens = [t for t in text.split() if len(t) > 1 and t not in DOMAIN_STOPWORDS]
    return WS_RE.sub(" ", " ".join(tokens)).strip()


def first_words(value, n=140):
    return " ".join(str(value).split()[:n])


def ranking_metrics(true_leafs, ranked_leafs, ks=(1, 3, 5)):
    true_leafs = list(map(str, true_leafs))
    metrics = {}
    for k in ks:
        metrics[f"top{k}_accuracy"] = float(np.mean([
            true in list(map(str, preds[:k]))
            for true, preds in zip(true_leafs, ranked_leafs)
        ]))
    reciprocal_ranks = []
    for true, preds in zip(true_leafs, ranked_leafs):
        preds = list(map(str, preds))
        reciprocal_ranks.append(1.0 / (preds.index(true) + 1) if true in preds else 0.0)
    metrics["mrr"] = float(np.mean(reciprocal_ranks))
    return metrics


def classification_metrics(y_true, y_pred, prefix="test"):
    return {
        f"{prefix}_accuracy": float(accuracy_score(y_true, y_pred)),
        f"{prefix}_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        f"{prefix}_weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }


def top10_leaf_macro_f1(y_true, y_pred):
    top10 = pd.Series(y_true).value_counts().head(10).index.astype(str).tolist()
    mask = pd.Series(y_true).isin(top10).to_numpy()
    return float(f1_score(np.asarray(y_true)[mask], np.asarray(y_pred)[mask], labels=top10, average="macro", zero_division=0))


def print_metrics(title, metrics):
    print(f"\n{title}")
    for key, value in metrics.items():
        if isinstance(value, (float, np.floating, int, np.integer)):
            print(f"{key:28s} {float(value):.4f}")
        else:
            print(f"{key:28s} {value}")

## Load Task 2, Merge LongProfile for Task 1, and Reuse Existing Split

In [7]:
for path in [TASK2_PATH, TASK1_SOURCE_PATH, TAXONOMY_PATH, SPLIT_PATH]:
    assert path.exists(), path

task2 = pd.read_csv(TASK2_PATH).reset_index().rename(columns={"index": "row_index"})
task2["leaf"]   = task2["SubIndustry"].map(lambda x: clean_code(x, width=10))
task2["parent"] = task2["leaf"].str[:8]
task2["SegmentName"]        = task2["SegmentName"].fillna("").map(normalize_text)
task2["SegmentDescription"] = task2["SegmentDescription"].fillna("").map(normalize_text)

task1_source = pd.read_csv(
    TASK1_SOURCE_PATH,
    usecols=["CompanyId", "AsOfDate", "LongProfile", "MstarGlobal"],
    dtype={"CompanyId": str, "AsOfDate": str},
)
task1_source["LongProfile"] = task1_source["LongProfile"].fillna("").map(normalize_text)

exact_profile = (
    task1_source.sort_values(["CompanyId", "AsOfDate"])
    .drop_duplicates(["CompanyId", "AsOfDate"])[["CompanyId", "AsOfDate", "LongProfile"]]
)
company_profile = (
    task1_source[task1_source["LongProfile"].ne("")]
    .drop_duplicates(["CompanyId"])[["CompanyId", "LongProfile"]]
    .rename(columns={"LongProfile": "LongProfile_company"})
)

task2 = task2.merge(exact_profile,   on=["CompanyId", "AsOfDate"], how="left")
task2 = task2.merge(company_profile, on="CompanyId",               how="left")
task2["LongProfile"] = task2["LongProfile"].fillna(task2["LongProfile_company"]).fillna("")
task2 = task2.drop(columns=["LongProfile_company"])

splits = pd.read_csv(SPLIT_PATH, dtype={"leaf": str, "parent": str})
assert len(splits) == len(task2), "Split length mismatch"
assert (splits["leaf"].astype(str).to_numpy() == task2["leaf"].astype(str).to_numpy()).all(), "Split leaf order mismatch"
task2["split"] = splits["split"].to_numpy()

# segment_text uses [SEGMENT_NAME] / [SEGMENT_DESCRIPTION] tokens — must match inference.py
task2["segment_text"] = (
    "[SEGMENT_NAME] " + task2["SegmentName"].fillna("")
    + " [SEGMENT_DESCRIPTION] " + task2["SegmentDescription"].fillna("")
)

if EVAL_ROW_LIMIT is not None:
    val_df  = task2[task2["split"].eq("val")].sample(n=min(EVAL_ROW_LIMIT,  task2["split"].eq("val").sum()),  random_state=SEED).reset_index(drop=True)
    test_df = task2[task2["split"].eq("test")].sample(n=min(EVAL_ROW_LIMIT, task2["split"].eq("test").sum()), random_state=SEED).reset_index(drop=True)
else:
    val_df  = task2[task2["split"].eq("val")].reset_index(drop=True)
    test_df = task2[task2["split"].eq("test")].reset_index(drop=True)

print(f"Task 2 rows: {len(task2):,}")
print(task2["split"].value_counts().to_string())
print(f"LongProfile coverage: {task2['LongProfile'].ne('').mean():.4f}")
print(f"Val: {len(val_df):,}  Test: {len(test_df):,}")


Task 2 rows: 27,537
split
train    16691
test      5425
val       5421
LongProfile coverage: 1.0000
Val: 5,421  Test: 5,425


## Load Taxonomy and Candidate Maps

In [8]:
taxonomy = pd.read_csv(TAXONOMY_PATH, dtype=str).fillna("")
taxonomy["industry_code"] = taxonomy["industry_code"].map(lambda x: clean_code(x, width=8))
taxonomy["subindustry_code"] = taxonomy["subindustry_code"].map(lambda x: clean_code(x, width=10))

taxonomy_text_by_leaf = taxonomy.set_index("subindustry_code")["taxonomy_text"].to_dict()
leaf_to_parent = taxonomy.set_index("subindustry_code")["industry_code"].to_dict()
leaf_name = taxonomy.set_index("subindustry_code")["subindustry_name"].to_dict()
parent_name = taxonomy.drop_duplicates("industry_code").set_index("industry_code")["industry_name"].to_dict()
leaves_by_parent = taxonomy.groupby("industry_code")["subindustry_code"].apply(lambda x: list(dict.fromkeys(x))).to_dict()
all_parent_codes = sorted(leaves_by_parent)

missing_leafs = sorted(set(task2["leaf"].unique()) - set(taxonomy_text_by_leaf))
missing_parents = sorted(set(task2["parent"].unique()) - set(leaves_by_parent))
print("taxonomy leaf classes:", len(taxonomy_text_by_leaf))
print("taxonomy parent classes:", len(leaves_by_parent))
print("missing leaf labels from taxonomy:", len(missing_leafs), missing_leafs[:10])
print("missing parent labels from taxonomy:", len(missing_parents), missing_parents[:10])

taxonomy leaf classes: 445
taxonomy parent classes: 145
missing leaf labels from taxonomy: 11 ['1013002011', '1028006009', '1035002002', '1041001002', '1042001001', '2072002005', '2072002006', '2072002007', '2072003003', '3102004003']
missing parent labels from taxonomy: 0 []


## v11 FLANG-BERT Parent Candidate Generator

In [9]:
import pickle, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy.special import log_softmax, softmax
from tqdm.auto import tqdm

TASK1_NUMERIC_COLS = [
    'revenue_share', 'is_largest_bin', 'log_revenue',
    'log_total_revenue', 'n_segments', 'herfindahl_index',
    'report_quarter', 'lp_short_flag', 'sd_short_flag', 'sn_short_flag',
]
TASK1_CONTINUOUS = [
    'revenue_share', 'log_revenue', 'log_total_revenue',
    'n_segments', 'herfindahl_index', 'report_quarter',
]


class FLANGMultiTask(nn.Module):
    def __init__(self, model_name, n_leaf, n_sector, n_group, num_features=10, dropout=0.1):
        super().__init__()
        self.bert        = AutoModel.from_pretrained(model_name, local_files_only=LOCAL_FILES_ONLY)
        h                = self.bert.config.hidden_size
        self.dropout     = nn.Dropout(dropout)
        self.leaf_head   = nn.Linear(h + num_features, n_leaf)
        self.sector_head = nn.Linear(h, n_sector)
        self.group_head  = nn.Linear(h, n_group)

    def forward(self, input_ids, attention_mask, numeric_feats):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        pooled = self.dropout(pooled)
        combined = torch.cat([pooled, numeric_feats.to(pooled.dtype)], dim=-1)
        return self.leaf_head(combined), self.sector_head(pooled), self.group_head(pooled)


class _TextDS(Dataset):
    def __init__(self, texts, tok, max_len):
        self.texts = texts; self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], truncation=True, padding='max_length',
                     max_length=self.max_len, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in e.items()}


def build_v11_text(frame):
    texts = []
    sib_lookup = {}
    if 'CompanyId' in frame.columns and 'AsOfDate' in frame.columns:
        for (cid, dt), grp in frame.groupby(['CompanyId', 'AsOfDate']):
            sib_lookup[(cid, dt)] = grp.index.tolist()
    for idx in frame.index:
        row      = frame.loc[idx]
        seg_name = str(row.get('SegmentName', '')).strip()
        seg_desc = str(row.get('SegmentDescription', '')).strip()
        long_p   = str(row.get('LongProfile',        '')).strip()
        year     = str(row.get('AsOfDate', ''))[:4]
        lp_short = ' '.join(long_p.split()[:100]) if long_p else ''
        sib_parts = []
        if 'CompanyId' in frame.columns:
            key = (row['CompanyId'], row['AsOfDate'])
            for sib_idx in sib_lookup.get(key, []):
                if sib_idx == idx: continue
                sib       = frame.loc[sib_idx]
                rev_pct   = round(float(sib.get('revenue_share', 0) or 0) * 100, 1)
                sib_d     = str(sib.get('SegmentDescription', '')).strip()
                sib_short = ' '.join(sib_d.split()[:25])
                sib_parts.append(f"[SEG {rev_pct}%] {sib_short}")
        parts = ["[]", "[]", f"[{year}]", "[PRIMARY]", seg_name, "[SEP]", seg_desc]
        if sib_parts: parts.extend(sib_parts)
        if lp_short:  parts.append(f"[LP] {lp_short}")
        texts.append(' '.join(parts))
    return texts


def compute_v11_numeric(frame, scaler):
    df = frame.copy()
    def to_f(s):
        try: return float(s)
        except: return 0.0
    df['Revenue']                     = df.get('Revenue',                     pd.Series(0, index=df.index)).apply(to_f)
    df['total_revenue_company_as_of'] = df.get('total_revenue_company_as_of', pd.Series(0, index=df.index)).apply(to_f)
    df['revenue_share']               = df.get('revenue_share',               pd.Series(0, index=df.index)).apply(to_f)
    df['log_revenue']       = np.log1p(df['Revenue'].clip(lower=0))
    df['log_total_revenue'] = np.log1p(df['total_revenue_company_as_of'].clip(lower=0))
    def _largest(v):
        if isinstance(v, str): return 1 if v.lower() in ('true','1','yes') else 0
        return int(bool(v))
    df['is_largest_bin'] = df.get('is_largest_share_segment', pd.Series(0, index=df.index)).apply(_largest)
    if 'CompanyId' in df.columns and 'AsOfDate' in df.columns:
        g = ['CompanyId', 'AsOfDate']
        df['n_segments']       = df.groupby(g)['SegmentName'].transform('count').astype(float)
        df['herfindahl_index'] = df.groupby(g)['revenue_share'].transform(lambda x: float((x**2).sum()))
    else:
        df['n_segments']       = 1.0
        df['herfindahl_index'] = (df['revenue_share']**2).fillna(0.0)
    def _qtr(d):
        try: return (int(str(d)[5:7]) - 1) // 3 + 1
        except: return 0
    df['report_quarter'] = df.get('AsOfDate', pd.Series('', index=df.index)).apply(_qtr).astype(float)
    def _short(t, n): return 1 if pd.isna(t) or len(str(t).split()) < n else 0
    df['lp_short_flag'] = df.get('LongProfile',        pd.Series('', index=df.index)).apply(lambda t: _short(t, 20))
    df['sd_short_flag'] = df.get('SegmentDescription', pd.Series('', index=df.index)).apply(lambda t: _short(t, 8))
    df['sn_short_flag'] = df.get('SegmentName',        pd.Series('', index=df.index)).apply(lambda t: _short(t, 3))
    for col in TASK1_NUMERIC_COLS:
        if col not in df.columns: df[col] = 0.0
        df[col] = df[col].fillna(0.0).astype(float)
    feats = df[TASK1_NUMERIC_COLS].copy()
    feats[TASK1_CONTINUOUS] = scaler.transform(feats[TASK1_CONTINUOUS])
    return feats[TASK1_NUMERIC_COLS].values.astype(np.float32)


@torch.no_grad()
def v11_log_scores(frame):
    with open(TASK1_V11_LABEL_ENC,  'rb') as f: le      = pickle.load(f)
    with open(TASK1_V11_SECTOR_ENC, 'rb') as f: le_sec  = pickle.load(f)
    with open(TASK1_V11_GROUP_ENC,  'rb') as f: le_grp  = pickle.load(f)
    with open(TASK1_V11_SCALER,     'rb') as f: scaler  = pickle.load(f)

    texts   = build_v11_text(frame)
    numeric = compute_v11_numeric(frame, scaler)
    numeric_t = torch.from_numpy(numeric)

    tok   = AutoTokenizer.from_pretrained(TASK1_V11_MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    model = FLANGMultiTask(
        TASK1_V11_MODEL_NAME, len(le.classes_),
        len(le_sec.classes_), len(le_grp.classes_)
    ).to(DEVICE)
    state = torch.load(TASK1_V11_STATE, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    model.eval()

    ds     = _TextDS(texts, tok, 512)
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())
    chunks, cursor = [], 0
    for batch in tqdm(loader, desc="v11 T1 scoring"):
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        bs     = batch['input_ids'].shape[0]
        num_b  = numeric_t[cursor:cursor+bs].to(DEVICE)
        cursor += bs
        leaf_logits, _, _ = model(batch['input_ids'], batch['attention_mask'], num_b)
        chunks.append(leaf_logits.float().cpu().numpy())

    del model, tok
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    logits  = np.vstack(chunks)
    classes = np.array([str(c).strip().zfill(8) for c in le.classes_])
    return log_softmax(logits, axis=1), classes

print("v11 FLANG-BERT scorer ready.")


v11 FLANG-BERT scorer ready.


### v11 candidate generation (replaces SVC + SEC-BERT streams)

In [10]:
# Old SEC-BERT/SVC streams removed — v11 FLANG-BERT handles candidate generation.
print("Skipping old SVC/SEC-BERT cells.")


Skipping old SVC/SEC-BERT cells.


## Build and Save Task 1 Parent Candidates

In [11]:
task1_log_val,  task1_classes = v11_log_scores(val_df)
task1_log_test, _             = v11_log_scores(test_df)

assert set(task1_classes) >= set(task2["parent"].unique()), \
    "v11 classes don't cover Task 2 parents — check label_encoder.pkl provenance"


def parent_candidates_from_scores(frame, log_scores, split_name):
    probs = softmax(log_scores, axis=1)
    order = np.argsort(-probs, axis=1)
    rows  = []
    for i, (_, row) in enumerate(frame.iterrows()):
        top_idx = order[i, :max(PARENT_TOPK, FALLBACK_PARENT_TOPK)]
        codes   = task1_classes[top_idx].astype(str).tolist()
        scores  = probs[i, top_idx].astype(float).tolist()
        rows.append({
            "source_row_id":    i,
            "row_index":        int(row["row_index"]),
            "CompanyId":        row["CompanyId"],
            "true_parent":      row["parent"],
            "true_leaf":        row["leaf"],
            "top_parent_codes": json.dumps(codes),
            "top_parent_scores":json.dumps(scores),
            "top1_parent":      codes[0],
            "top1_parent_score":scores[0],
        })
    cand = pd.DataFrame(rows)
    cand.to_csv(OUTPUT_DIR / f"task1_parent_candidates_{split_name}.csv", index=False)
    return cand


def parent_topk_recall(cand, ks=(1, 3, 5, 10)):
    out    = {}
    parsed = cand["top_parent_codes"].map(json.loads)
    truth  = cand["true_parent"].astype(str).tolist()
    for k in ks:
        out[f"parent_top{k}_recall"] = float(np.mean(
            [t in codes[:k] for t, codes in zip(truth, parsed)]
        ))
    return out


start = time.time()
val_parent_candidates  = parent_candidates_from_scores(val_df,  task1_log_val,  "val")
test_parent_candidates = parent_candidates_from_scores(test_df, task1_log_test, "test")

parent_metrics = {
    "val":     parent_topk_recall(val_parent_candidates),
    "test":    parent_topk_recall(test_parent_candidates),
    "seconds": round(time.time() - start, 1),
}
with open(OUTPUT_DIR / "task1_parent_candidate_metrics.json", "w") as f:
    json.dump(parent_metrics, f, indent=2)
print(json.dumps(parent_metrics, indent=2))


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/369 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

v11 T1 scoring:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: SALT-NLP/FLANG-BERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


v11 T1 scoring:   0%|          | 0/340 [00:00<?, ?it/s]

{
  "val": {
    "parent_top1_recall": 0.7959786017339974,
    "parent_top3_recall": 0.9383877513373916,
    "parent_top5_recall": 0.9653200516509869,
    "parent_top10_recall": 0.9883785279468733
  },
  "test": {
    "parent_top1_recall": 0.7863594470046082,
    "parent_top3_recall": 0.9349308755760368,
    "parent_top5_recall": 0.9666359447004609,
    "parent_top10_recall": 0.9878341013824885
  },
  "seconds": 1.2
}


## Load Task 2 Cross-Encoder

In [12]:
def build_pair_text(segment_text, taxonomy_text):
    return (
        f"[SEGMENT] {normalize_text(segment_text)} "
        f"[CANDIDATE_TAXONOMY] {normalize_text(taxonomy_text)} "
        "[QUESTION] Does this segment belong to this subindustry?"
    ).strip()


class PairDataset(Dataset):
    def __init__(self, pair_df, tokenizer, max_length=MAX_LENGTH):
        self.texts     = pair_df["pair_text"].astype(str).tolist()
        self.labels    = pair_df["label"].astype(np.float32).to_numpy()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc  = self.tokenizer(self.texts[idx], truncation=True,
                              max_length=self.max_length, padding="max_length",
                              return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


def masked_mean_pool(last_hidden_state, attention_mask):
    mask   = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom  = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom


class SegmentTaxonomyCrossEncoder(nn.Module):
    def __init__(self, model_name=TASK2_CROSS_ENCODER_MODEL, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, local_files_only=LOCAL_FILES_ONLY)
        self.encoder.float()
        hidden_size      = self.encoder.config.hidden_size
        self.dropout     = nn.Dropout(dropout)
        self.classifier  = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        if getattr(outputs, "pooler_output", None) is not None:
            pooled = outputs.pooler_output
        else:
            pooled = masked_mean_pool(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        pooled = pooled.to(self.classifier.weight.dtype)
        return self.classifier(pooled).squeeze(-1)


@torch.no_grad()
def predict_pair_logits(model, pair_df, tokenizer, batch_size=EVAL_BATCH_SIZE):
    ds     = PairDataset(pair_df, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())
    logits = []
    model.eval()
    for batch in tqdm(loader, desc="T2 cross-encoder scoring", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out   = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        logits.append(out.float().cpu().numpy())
    return np.concatenate(logits)

print("Cross-encoder classes loaded.")


Cross-encoder classes loaded.


## Build Candidate Subindustries and Score Them

In [13]:
def candidate_leafs_from_parent_codes(parent_codes, min_candidates=MIN_CANDIDATE_LEAFS):
    seen = []
    for parent in parent_codes:
        for leaf in leaves_by_parent.get(str(parent), []):
            if leaf not in seen:
                seen.append(leaf)
    if len(seen) < min_candidates:
        frequent_leafs = task2["leaf"].value_counts().index.astype(str).tolist()
        for leaf in frequent_leafs:
            if leaf not in seen:
                seen.append(leaf)
            if len(seen) >= min_candidates:
                break
    return seen


def candidate_rows_for_frame(frame, parent_candidates, split_name):
    parent_by_pos = {int(row.source_row_id): row for row in parent_candidates.itertuples(index=False)}
    rows = []
    for pos, row in frame.reset_index(drop=True).iterrows():
        cand_row = parent_by_pos[pos]
        parent_codes = json.loads(cand_row.top_parent_codes)
        parent_scores = json.loads(cand_row.top_parent_scores)
        use_k = PARENT_TOPK
        if parent_scores and parent_scores[0] < LOW_CONFIDENCE_THRESHOLD:
            use_k = FALLBACK_PARENT_TOPK
        parent_codes = parent_codes[:use_k]
        parent_scores = parent_scores[:use_k]
        score_by_parent = {p: s for p, s in zip(parent_codes, parent_scores)}
        candidate_leafs = candidate_leafs_from_parent_codes(parent_codes)
        for leaf in candidate_leafs:
            parent = leaf[:8]
            tax_text = taxonomy_text_by_leaf.get(leaf, f"[SUBINDUSTRY_CODE] {leaf} [INDUSTRY_CODE] {parent}")
            rows.append({
                "source_row_id": pos,
                "row_index": int(row["row_index"]),
                "split": split_name,
                "true_leaf": row["leaf"],
                "true_parent": row["parent"],
                "candidate_leaf": leaf,
                "candidate_parent": parent,
                "candidate_parent_name": parent_name.get(parent, ""),
                "candidate_leaf_name": leaf_name.get(leaf, ""),
                "parent_score": float(score_by_parent.get(parent, EPS)),
                "pair_text": build_pair_text(row["segment_text"], tax_text),
                "label": int(leaf == row["leaf"]),
            })
    return pd.DataFrame(rows)


def score_split(frame, parent_candidates, split_name):
    pair_df = candidate_rows_for_frame(frame, parent_candidates, split_name)
    tokenizer = AutoTokenizer.from_pretrained(TASK2_CROSS_ENCODER_MODEL, local_files_only=LOCAL_FILES_ONLY)
    model = SegmentTaxonomyCrossEncoder().to(DEVICE)
    model.load_state_dict(torch.load(CROSS_ENCODER_DIR / "best_state.pt", map_location=DEVICE))
    pair_df["cross_encoder_logit"] = predict_pair_logits(model, pair_df.assign(label=0), tokenizer)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pair_df


start = time.time()
val_scored = score_split(val_df, val_parent_candidates, "val")
test_scored = score_split(test_df, test_parent_candidates, "test")
val_scored.to_csv(OUTPUT_DIR / "scored_pairs_val_task1_best_topk.csv", index=False)
test_scored.to_csv(OUTPUT_DIR / "scored_pairs_test_task1_best_topk.csv", index=False)
print(f"Scoring seconds: {time.time() - start:.1f}")
print("avg val candidates:", round(val_scored.groupby("source_row_id").size().mean(), 2))
print("avg test candidates:", round(test_scored.groupby("source_row_id").size().mean(), 2))
print("val candidate coverage:", val_scored.groupby("source_row_id")["label"].max().mean())
print("test candidate coverage:", test_scored.groupby("source_row_id")["label"].max().mean())

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

T2 cross-encoder scoring:   0%|          | 0/11791 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


T2 cross-encoder scoring:   0%|          | 0/11719 [00:00<?, ?it/s]

Scoring seconds: 701.1
avg val candidates: 34.8
avg test candidates: 34.56
val candidate coverage: 0.9572034679948349
test candidate coverage: 0.9557603686635945


## Tune Parent Prior Alpha on Validation and Evaluate Test

In [14]:
def rank_scored_pairs(scored_pairs, alpha):
    df = scored_pairs.copy()
    df["final_score"] = df["cross_encoder_logit"] + alpha * np.log(df["parent_score"].clip(EPS, 1.0))
    ranked = []
    truth = []
    best_rows = []
    for row_id, group in df.groupby("source_row_id", sort=True):
        group = group.sort_values("final_score", ascending=False)
        ranked.append(group["candidate_leaf"].astype(str).tolist())
        truth.append(str(group["true_leaf"].iloc[0]))
        best_rows.append(group.iloc[0].to_dict())
    pred = [r[0] if r else "" for r in ranked]
    return truth, ranked, np.array(pred, dtype=str), pd.DataFrame(best_rows)


def evaluate_scored_pairs(scored_pairs, alpha, prefix):
    truth, ranked, pred, best_rows = rank_scored_pairs(scored_pairs, alpha)
    metrics = ranking_metrics(truth, ranked, ks=(1, 3, 5))
    metrics.update(classification_metrics(np.array(truth), pred, prefix=prefix))
    metrics[f"{prefix}_top10_leaf_macro_f1"] = top10_leaf_macro_f1(np.array(truth), pred)
    metrics["avg_candidates"] = float(scored_pairs.groupby("source_row_id").size().mean())
    metrics["candidate_coverage"] = float(scored_pairs.groupby("source_row_id")["label"].max().mean())
    return metrics, best_rows


rows = []
for alpha in ALPHA_GRID:
    metrics, _ = evaluate_scored_pairs(val_scored, alpha, prefix="val")
    rows.append({"alpha": alpha, **metrics})
alpha_sweep = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
alpha_sweep.to_csv(OUTPUT_DIR / "alpha_sweep.csv", index=False)
display(alpha_sweep)

best_alpha = float(alpha_sweep.iloc[0]["alpha"])
val_metrics, val_best = evaluate_scored_pairs(val_scored, best_alpha, prefix="val")
test_metrics, test_best = evaluate_scored_pairs(test_scored, best_alpha, prefix="test")
val_best.to_csv(OUTPUT_DIR / "best_predictions_val.csv", index=False)
test_best.to_csv(OUTPUT_DIR / "best_predictions_test.csv", index=False)

final_rows = [
    {"split": "val", "alpha": best_alpha, **val_metrics},
    {"split": "test", "alpha": best_alpha, **test_metrics},
]
final_metrics = pd.DataFrame(final_rows)
final_metrics.to_csv(OUTPUT_DIR / "task1_best_pipeline_metrics.csv", index=False)

with open(OUTPUT_DIR / "task1_best_pipeline_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_alpha": best_alpha,
        "val": val_metrics,
        "test": test_metrics,
        "parent_candidate_metrics": parent_metrics,
    }, f, indent=2)

print_metrics("VAL best", {"alpha": best_alpha, **val_metrics})
print_metrics("TEST best", {"alpha": best_alpha, **test_metrics})
display(final_metrics)

,alpha,top1_accuracy,top3_accuracy,top5_accuracy,mrr,val_accuracy,val_macro_f1,val_weighted_f1,val_top10_leaf_macro_f1,avg_candidates,candidate_coverage
7,7.5,0.665744,0.839697,0.876960,0.761340,0.665744,0.532832,0.648578,0.839889,34.80059,0.957203
8,10.0,0.665744,0.831950,0.868659,0.758278,0.665744,0.531849,0.648901,0.839301,34.80059,0.957203
9,15.0,0.662608,0.823464,0.861280,0.753582,0.662608,0.529425,0.646875,0.830535,34.80059,0.957203
6,5.0,0.662239,0.845600,0.885814,0.762729,0.662239,0.526110,0.643669,0.843905,34.80059,0.957203
5,3.0,0.649142,0.850028,0.893193,0.756412,0.649142,0.515232,0.626882,0.845178,34.80059,0.957203
4,2.0,0.620365,0.838406,0.896514,0.738150,0.620365,0.491618,0.595507,0.839036,34.80059,0.957203
3,1.5,0.593064,0.823833,0.893747,0.718417,0.593064,0.462502,0.567840,0.833790,34.80059,0.957203
2,1.0,0.570374,0.806493,0.892640,0.700464,0.570374,0.444033,0.544989,0.827347,34.80059,0.957203
1,0.5,0.541413,0.782881,0.876407,0.677147,0.541413,0.426586,0.516652,0.793392,34.80059,0.957203
0,0.0,0.425383,0.708172,0.808338,0.589708,0.425383,0.330697,0.406492,0.655270,34.80059,0.957203



VAL best
alpha                        7.5000
top1_accuracy                0.6657
top3_accuracy                0.8397
top5_accuracy                0.8770
mrr                          0.7613
val_accuracy                 0.6657
val_macro_f1                 0.5328
val_weighted_f1              0.6486
val_top10_leaf_macro_f1      0.8399
avg_candidates               34.8006
candidate_coverage           0.9572

TEST best
alpha                        7.5000
top1_accuracy                0.6581
top3_accuracy                0.8315
top5_accuracy                0.8721
mrr                          0.7546
test_accuracy                0.6581
test_macro_f1                0.5106
test_weighted_f1             0.6395
test_top10_leaf_macro_f1     0.8043
avg_candidates               34.5624
candidate_coverage           0.9558


,split,alpha,top1_accuracy,top3_accuracy,top5_accuracy,mrr,val_accuracy,val_macro_f1,val_weighted_f1,val_top10_leaf_macro_f1,avg_candidates,candidate_coverage,test_accuracy,test_macro_f1,test_weighted_f1,test_top10_leaf_macro_f1
0,val,7.5,0.665744,0.839697,0.876960,0.761340,0.665744,0.532832,0.648578,0.839889,34.800590,0.957203,NaN,NaN,NaN,NaN
1,test,7.5,0.658065,0.831521,0.872074,0.754644,NaN,NaN,NaN,NaN,34.562396,0.955760,0.658065,0.510592,0.639541,0.804263


In [15]:
import shutil, json
from pathlib import Path

OUTPUT_DIR_DRIVE = BASE_DIR / 'task2_assets' / 'task2_v11_outputs'
OUTPUT_DIR_DRIVE.mkdir(parents=True, exist_ok=True)

LOCAL_OUT = Path('models/task2_task1_best_candidate_reranker')

for fname in LOCAL_OUT.rglob('*'):
    if fname.is_file():
        dest = OUTPUT_DIR_DRIVE / fname.relative_to(LOCAL_OUT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(fname, dest)
        print(f"saved  {fname.name}")

print(f"\nAll outputs saved to Drive: {OUTPUT_DIR_DRIVE}")


All outputs saved to Drive: /content/drive/MyDrive/CAPSTONE/task2_assets/task2_v11_outputs
